# 05 · MD system setup — PSMA / GCPII (Q04609)

**Milestone 6.** Everything stays inside Colab.
- **Part A (sections B–E):** bonded force-field parameters for the binuclear zinc site with **MCPB.py** (AmberTools); the quantum chemistry runs in **PySCF**. Design: [`docs/M6_ZINC_PARAMETERS.md`](../docs/M6_ZINC_PARAMETERS.md).
- **Part B (sections F–J):** the full membrane system (both chains, lipids, water, salt), built with packmol-memgen and tleap, then minimised and test-run in OpenMM. Design: [`docs/M6_SYSTEM_BUILD.md`](../docs/M6_SYSTEM_BUILD.md).

Part B reuses part A's files from Drive, so once part A has finished you can re-run the notebook and it skips straight through (finished QM steps are cached).

| Decision | Choice | Why |
|---|---|---|
| Zn treatment | **bonded (MCPB.py)** | your choice: keeps the coordination geometry |
| Bridging species | **hydroxide**, Glu424 protonated | GCPII QM/MM studies (Plechanovová *et al.* 2011, *J Med Chem* 54:7535) |
| QM program | **PySCF**, Gaussian-equivalent B3LYP/6-31G* (`b3lypg`, UltraFine grid) | free, runs in Colab, uses the GPU if possible |
| Protein force field / water | ff14SB / TIP3P | the validated combination with Lipid21 membranes |
| Metal–ligand cut-off | 2.5 Å | gives the literature coordination of 9S7U: E425 bidentate; D453 OD1 and a weak extra water excluded |
| Sites | chain A parameterised; chain B reuses the templates | the two sites are identical |

**How it works.** MCPB.py builds the cluster models and writes its usual Gaussian inputs. The `qm_bridge` runs those *same* inputs with PySCF and writes the files MCPB.py reads (the `.fchk` and the ESP `.log`). MCPB.py then derives the force constants (Seminario method), the RESP charges and the tleap input exactly as usual.

**Runtime.** Use a **GPU runtime** if you can. The geometry optimisation and Hessian of the ~50-atom small model take roughly 15–60 min on a GPU and several hours on CPU only. Results are saved to Drive after each step, so a disconnected run **resumes** from the last finished step.

> **Setup (section 0).** AmberTools, PySCF and OpenMM are installed into a **separate environment with its own Python 3.11**, because Colab's Python (3.13) is newer than AmberTools' builds. This takes about 5–10 min once per session and needs **no runtime restart**. The heavy stages run in that environment; this notebook only shows the results.

## 0 · Setup

In [7]:
#@title 0a · the pipeline (Colab's own Python: display only)
import os, sys, subprocess, shutil, time
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
USE_GPU = True    # PySCF on the GPU via gpu4pyscf; falls back to CPU automatically
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(Path("src").resolve()))
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
print("psma_pipeline", psma_pipeline.__version__, "| Python", sys.version.split()[0])

psma_pipeline 0.7.6 | Python 3.13.16


In [8]:
#@title 0b · MD environment: AmberTools + packmol + PySCF + OpenMM (Python 3.11, ~5–10 min once per session)
from psma_pipeline import md_env
t0 = time.time()
info = md_env.install_env(Path(".").resolve(), gpu=USE_GPU)
print(f"ready in {(time.time() - t0) / 60:.1f} min")
for k, v in info.items():
    print(f"  {k}: {v}")
assert md_env.env_ready(), "MD environment incomplete: see /content/mdenv_install.log"

PySCF will use the GPU: gpu4pyscf 1.9.1, 1 GPU(s)
ready in 0.1 min
  machine: Colab
  NVIDIA GPU: NVIDIA A100-SXM4-80GB, 580.82.07
  prefix: /content/mdenv
  python: /content/mdenv/bin/python
  gpu4pyscf installed: True
  MCPB.py Seminario patch: True
  install log: /content/mdenv_install.log
  pyscf: 2.15.0
  openmm: 8.6.1
  pandas: 3.0.6
  numpy: 2.4.6
  openmm platforms: ['Reference', 'CPU', 'CUDA', 'OpenCL']
  PySCF on GPU: [True, 'gpu4pyscf 1.9.1, 1 GPU(s)']


## 1 · Configuration and the prepared model (notebook 04)

In [9]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import storage_overrides
from psma_pipeline.sanity import critical_failures

setup_logging()
cfg = load_config(overrides=storage_overrides(True) or None)
OUT, CACHE = cfg.outputs_dir, cfg.cache_dir
GPU_ARGS = [] if USE_GPU else ["--no-gpu"]
prep = cfg.outputs_dir / "model_preparation" / cfg.get("preparation.name") / "microstates" / "microstate_00_default.pdb"
assert prep.exists(), f"{prep} not found: run notebook 04 first"
print("prepared model:", prep)
print("Zn treatment:", cfg.get("preparation.zn_md_treatment"), "| bridge:", cfg.get("preparation.zn_bridge"),
      "| protein FF:", cfg.get("md.protein_ff"), "| cut-off:", cfg.get("md.mcpb.cut_off"), "Å")
assert cfg.get("preparation.zn_md_treatment") == "bonded_mcpb", "this notebook builds the bonded (MCPB.py) model"

Results and downloads are stored on Google Drive: /content/drive/MyDrive/psma_structural_modelling
prepared model: /content/drive/MyDrive/psma_structural_modelling/outputs/model_preparation/hybrid_9S7U_AF2/microstates/microstate_00_default.pdb
Zn treatment: bonded_mcpb | bridge: hydroxide | protein FF: ff14SB | cut-off: 2.5 Å


## A · Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "Unit tests failed: fix before running the QM."

## B · QM self-test (water, about a minute)

Before the long zinc calculation, the PySCF bridge is tested end to end on water at B3LYP/6-31G*:
- the harmonic frequencies must match the published values (1713, 3727 and 3849 cm⁻¹) within 25 cm⁻¹;
- the `.fchk` and ESP `.log` it writes must be read back correctly by **AmberTools' own MCPB.py parsers**.

In [ ]:
st = md_env.run_stage("selftest", OUT, CACHE, extra=GPU_ARGS)
if "SelfTest" in st["tables"]:
    display(st["tables"]["SelfTest"])
assert st["returncode"] == 0, "QM self-test failed: do not run the metal-site QM until this passes"

## C · Bonded zinc parameters (MCPB.py + PySCF)

The steps:
1. **Site PDB**: chain A of the prepared model, with hydrogens added by tleap; the two Zn; the bridging water converted to a hydroxide (`OHX`).
2. **MCPB.py step 1** writes the small-model and large-model Gaussian inputs.
3. **PySCF** runs them:
   - small model: optimisation, then the analytic Hessian;
   - large model: the ESP on Merz–Kollman points.
4. **MCPB.py steps 2–4**: the Seminario force constants, RESP charges and tleap input.
5. **Validation build** of the parameterised chain with tleap.

> **Gate.** The run stops on a critical failure:
> - QM not converged;
> - tleap errors or missing parameters;
> - the metal-site charge is not an integer;
> - a Zn bond without parameters;
> - an original file modified.

Files are written to `outputs/md_setup/zinc_mcpb/` on Drive. Re-running resumes: finished QM steps are reused.

In [ ]:
res = md_env.run_stage("zinc", OUT, CACHE, extra=GPU_ARGS)
T = res["tables"]
print(f"finished in {res['extra'].get('minutes')} min; work folder: {res['extra'].get('workdir')}")
display(T.get("Issues"))
if "ZincChecks" in T:
    display(T["ZincChecks"])
assert res["returncode"] == 0, "zinc parameters failed: see the output above, ZincChecks and the logs in the work folder"

## D · Results

**Zn–ligand bonds.** Each bond is shown with its experimental distance (from 9S7U, as in the prepared model), the QM equilibrium length r0 and the force constant k.
- r0 within about ±0.1 Å of experiment is expected.
- Typical force constants are 30–100 kcal/mol/Å².

In [ ]:
display(T["SiteGeometry"]); display(T["QMSummary"])
zb = T["ZnBonds"]
display(zb)
if len(zb):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    lab = zb["Atom 2"] + " – " + zb["Atom 1"].str.split(":").str[0]
    x = np.arange(len(zb))
    ax[0].bar(x - 0.2, zb["Model distance (Å)"], 0.4, label="experimental (9S7U)")
    ax[0].bar(x + 0.2, zb["MCPB r0 (Å)"], 0.4, label="QM r0 (MCPB)")
    ax[0].set(xticks=x, ylabel="Å", ylim=(1.6, 2.6), title="Zn–ligand bond lengths"); ax[0].set_xticklabels(lab, rotation=60, ha="right", fontsize=8)
    ax[0].legend(fontsize=8)
    ax[1].bar(x, zb["MCPB k (kcal/mol/Å²)"], color="tab:green")
    ax[1].set(xticks=x, ylabel="kcal/mol/Å²", title="Seminario force constants"); ax[1].set_xticklabels(lab, rotation=60, ha="right", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
display(T.get("FrcmodAngl"))
display(T["SiteResidues"])
display(T["TleapCheck"])

## E · Saved files

In [ ]:
# the stage already saved its workbook, tables and manifest to Drive:
for k, v in res["extra"].get("saved", {}).items():
    print(f"{k}: {v}")

# Part B · Membrane system, minimisation and MD readiness

**What gets built** (all choices are in `config/default_config.yaml`, section `md.system` / `md.openmm`):

| Component | Choice | Status |
|---|---|---|
| Protein | both chains of the hybrid dimer, membrane-oriented (notebook 04), ff14SB | |
| Zinc sites | both sites with the part A MCPB.py parameters (chain B reuses chain A's templates, matched by ligand residues) | |
| Glycans | core GlcNAc at the observed sites, GLYCAM06j (`0YB` on `NLN`) | **assumption**: no full glycoforms |
| Membrane | **POPC : cholesterol 7 : 3**, Lipid21 | **assumption**: placeholder, to compare with a CHARMM-GUI build |
| Water / salt | TIP3P, 0.15 M NaCl, neutralised | |

**Steps.**
1. The **solute** is assembled: MCPB.py residue names applied to both sites, hydroxides, GLYCAM GlcNAc, Ca²⁺/Cl⁻ and crystal waters.
2. **packmol-memgen** packs the membrane and water around the protein (pre-oriented; it never protonates or changes the protein).
3. The solute is superposed back onto the packed protein; any lipid, water or ion **molecule** that touches it is removed whole.
4. **tleap** builds the parameter files (`system.prmtop` / `system.inpcrd`) and neutralises.
5. **OpenMM** minimises (first with the protein held, then free) and runs a short restrained smoke test (20 ps NVT heating + 20 ps NPT with the membrane barostat) on the GPU.

**Runtime.** Packing (CPU only): about 1–2 h for this ~520k-atom system; it runs on a **CPU runtime** too, then minimisation and the test run follow when a GPU is available (about 20–40 min). The packed membrane is cached on Drive; delete `outputs/md_setup/system/packed.pdb` to repack.

## F · Build the system

In [ ]:
s = cfg.get("md.system")
print(f"lipids {s['lipids']} ({s['ratio']}), salt {s['salt_molar']} M, water layer {s['dist_wat']} Å, clash cut-off {s['clash_cutoff']} Å")
HAS_GPU = md_env.nvidia_gpu() is not None
MINIMISE_ON_CPU = False  # True: minimise on the CPU without a GPU (very slow for this ~535k-atom system: ~40 s per step on
                         # a Colab CPU; saved after every chunk, resumable). Default: build only, minimise on a GPU;
                         # the 40 ps test run always needs a GPU and is reported as NOT RUN until then
if HAS_GPU:
    extra = []
elif MINIMISE_ON_CPU:
    print("No GPU: building, then minimising on the CPU (slow; progress is saved to Drive after every chunk,\n"
          "so re-running continues where it stopped). The 40 ps test run waits for a GPU runtime.")
    extra = []
else:
    print("No GPU: build only. Run this section again on a GPU runtime (packing and build are reused).")
    extra = ["--no-md"]
sysres = md_env.run_stage("system", OUT, CACHE, extra=extra)
S = sysres["tables"]
print(f"finished in {sysres['extra'].get('minutes')} min; work folder: {sysres['extra'].get('workdir')}")
display(S.get("Issues"))
assert "SystemChecks" in S, "system build stopped early: see the Issues table above and the logs in the work folder"


## G · What went into the box

- **SoluteSummary**: counts of chains, Zn, hydroxides, glycans, ions and crystal waters.
- **ZnTemplateMap**: each of the 4 Zn and the parameterised template it was matched to. Both chains should show the same two templates.
- **Composition**: molecules kept and removed. Only a handful of molecules should be removed for clashes.
- **MergeSummary**: superposition RMSD (should be ~0 Å: packmol-memgen does not move the protein) and the cholesterol fraction actually achieved.

In [ ]:
for k in ("SoluteSummary", "ZnTemplateMap", "GlycanLinks", "Composition", "MergeSummary", "TleapBuild"):
    if k in S:
        print(k); display(S[k])

## H · Minimisation and smoke run

What to look for:
- **Max force** must drop to below 1000 kJ/mol/nm after the unrestrained minimisation.
- **Zn–ligand distances** must stay within 0.3 Å of the experimental site after minimisation (the bonded model is working).
- **Temperature** in the smoke run should be close to 310 K, and the box should change only a little.

In [ ]:
if "Minimisation" in S:
    m = S["Minimisation"]
    display(m)
    if "MinimisationChunks" in S: display(S["MinimisationChunks"])
    if "RunawayForces" in S: display(S["RunawayForces"])
    if "FinalForces" in S: display(S["FinalForces"])
    ok = m.dropna(subset=["Potential energy (kJ/mol)"])
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
    ax[0].bar(ok["Stage"], ok["Max force (kJ/mol/nm)"]); ax[0].set_yscale("log")
    ax[0].axhline(cfg.get("md.openmm.max_force_ok"), color="r", ls="--", label="limit"); ax[0].legend()
    ax[0].set(title="Largest force on any atom", ylabel="kJ/mol/nm"); ax[0].tick_params(axis="x", rotation=30)
    if "Max Zn–ligand change (Å)" in ok:
        ax[1].bar(ok["Stage"], ok["Max Zn–ligand change (Å)"], color="tab:green")
    ax[1].axhline(cfg.get("md.openmm.zn_bond_max_change"), color="r", ls="--", label="limit (minimisation)")
    ax[1].set(title="Largest change of a Zn–ligand distance", ylabel="Å"); ax[1].tick_params(axis="x", rotation=30); ax[1].legend()
    plt.tight_layout(); plt.show()

## I · Checks and MD readiness

This is the first readiness table in the pipeline that can say **READY**: it does so only when no item FAILs. The two assumptions (lipid composition and glycans) are always listed as WARNING, so they stay visible in every report.

In [ ]:
display(S["SystemChecks"])
display(S["MDReadiness"])
assert sysres["returncode"] == 0, "critical check failed: see SystemChecks and the logs in the work folder"

## J · Saved files

In [ ]:
for k, v in sysres["extra"].get("saved", {}).items():
    print(f"{k}: {v}")

## Comparing with a CHARMM-GUI build later

To compare, build the same protein in CHARMM-GUI *Membrane Builder* (same PDB: `outputs/md_setup/system/protein_for_packing.pdb` shows the orientation used here; same lipids and salt), then compare:
- composition (lipids per leaflet, waters, ions) and box size, from **Composition** / **MergeSummary**;
- the minimised Zn–ligand distances (**Minimisation**);
- later, in equilibration: area per lipid, membrane thickness and protein tilt.

CHARMM-GUI cannot use the MCPB.py zinc model (it uses CHARMM36m), so differences in the zinc site are expected and are part of what the comparison tests.